# DS-GA 1011 HW4 — Part 1: Fine-tuning BERT
Run on Google Colab (GPU). Code lives in Google Drive (`dsga1011/nyu-dsga1011-a4/part1`). Cells below were collected from my working notebooks, with their original outputs.


## Setup

In [ ]:
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

from google.colab import drive
drive.mount('/content/drive')

GPU: True Tesla T4
Mounted at /content/drive


In [ ]:
import os
os.makedirs('/content/drive/MyDrive/dsga1011', exist_ok=True)
%cd /content/drive/MyDrive/dsga1011
if not os.path.exists('nyu-dsga1011-a4'):
    !git clone https://github.com/gregdurrett/nyu-dsga1011-a4
%cd nyu-dsga1011-a4/part1

/content/drive/MyDrive/dsga1011
/content/drive/MyDrive/dsga1011/nyu-dsga1011-a4/part1


In [ ]:
# Upgrade datasets and huggingface_hub to fix the HfUriError bug
!pip install -q --upgrade datasets huggingface_hub evaluate nltk

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [ ]:
!python -m nltk.downloader wordnet omw-1.4 punkt punkt_tab

<frozen runpy>:130: RuntimeWarning: 'nltk.downloader' found in sys.modules after import of package 'nltk', but prior to execution of 'nltk.downloader'; this may result in unpredictable behaviour
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Unzipping corpora/wordnet.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Unzipping corpora/omw-1.4.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
import transformers, datasets, evaluate
print(torch.__version__, transformers.__version__, datasets.__version__)
from nltk.corpus import wordnet
from nltk import word_tokenize
print(word_tokenize("The soup was hot."), wordnet.synsets("good")[:2])

2.14.1+cu130 5.22.1 5.2.0
['The', 'soup', 'was', 'hot', '.'] [Synset('good.n.01'), Synset('good.n.02')]


Newer `huggingface_hub` requires the full dataset id.

In [ ]:
%cd /content/drive/MyDrive/dsga1011/nyu-dsga1011-a4/part1
!sed -i 's/load_dataset("imdb")/load_dataset("stanfordnlp\/imdb")/' main.py
!grep -n 'load_dataset(' main.py

/content/drive/MyDrive/dsga1011/nyu-dsga1011-a4/part1
170:    dataset = load_dataset("stanfordnlp/imdb")


## Q1. Training loop (`do_train` in `main.py`)

In [42]:
path = "/content/drive/MyDrive/dsga1011/nyu-dsga1011-a4/part1/main.py"
src = open(path).read()

new_block = """    ################################
    ##### YOUR CODE BEGINGS HERE ###

    # Implement the training loop --- make sure to use the optimizer and lr_sceduler (learning rate scheduler)
    # Remember that pytorch uses gradient accumumlation so you need to use zero_grad (https://pytorch.org/tutorials/recipes/recipes/zeroing_out_gradients.html)
    # You can use progress_bar.update(1) to see the progress during training
    # You can refer to the pytorch tutorial covered in class for reference

    use_amp = device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    for epoch in range(num_epochs):
        correct, total, running_loss = 0, 0, 0.0

        for step, batch in enumerate(train_dataloader):
            batch = {k: v.to(device) for k, v in batch.items()}

            # trim padding to the longest real sequence in this batch
            max_len = int(batch["attention_mask"].sum(dim=1).max())
            batch = {k: (v[:, :max_len] if v.dim() == 2 else v) for k, v in batch.items()}

            optimizer.zero_grad()
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                outputs = model(**batch)
                loss = outputs.loss

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            lr_scheduler.step()

            preds = torch.argmax(outputs.logits, dim=-1)
            correct += (preds == batch["labels"]).sum().item()
            total += batch["labels"].size(0)
            running_loss += loss.item()

            progress_bar.update(1)
            progress_bar.set_postfix(
                epoch=epoch + 1,
                loss=f"{running_loss / (step + 1):.4f}",
                acc=f"{correct / total:.4f}",
            )

        print(f"Epoch {epoch + 1}/{num_epochs} | "
              f"train loss: {running_loss / len(train_dataloader):.4f} | "
              f"train acc: {correct / total:.4f}")

    ##### YOUR CODE ENDS HERE ######"""

s, e = src.index("def do_train("), src.index("def do_eval(")
fn = src[s:e]
b = fn.index("    ################################\n    ##### YOUR CODE BEGINGS HERE ###")
end_tag = "##### YOUR CODE ENDS HERE ######"
f = fn.index(end_tag) + len(end_tag)
src = src[:s] + fn[:b] + new_block + fn[f:] + src[e:]
open(path, "w").write(src)
print("Patched")
!grep -n "set_postfix\|GradScaler" {path}

Patched
49:    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
79:            progress_bar.set_postfix(


Debug run (target > 88%)

In [43]:
!python3 main.py --train --eval --debug_train

Debug training...
len(train_dataloader): 500
len(eval_dataloader): 125
Loading weights: 100% 199/199 [00:00<00:00, 5287.28it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params

Full run (target > 91%) → `out_original.txt`

In [44]:
!python3 main.py --train --eval

Actual training...
len(train_dataloader): 3125
len(eval_dataloader): 3125
Loading weights: 100% 199/199 [00:00<00:00, 11936.43it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those pa

In [45]:
import shutil, os
!ls -la out_original.txt out/
!tail -n 3 out_original.txt

sub = "/content/drive/MyDrive/dsga1011/hw4_submission"
os.makedirs(sub, exist_ok=True)
shutil.copy("out_original.txt", f"{sub}/out_original.txt")
print("Saved to", sub)

-rw------- 1 root root 100000 Nov 14 02:36 out_original.txt

out/:
total 423125
drwx------ 2 root root      4096 Nov 14 02:23 .
drwx------ 5 root root      4096 Nov 14 02:36 ..
-rw------- 1 root root       815 Nov 14 02:23 config.json
-rw------- 1 root root 433270744 Nov 14 02:23 model.safetensors
1
1
1
Saved to /content/drive/MyDrive/dsga1011/hw4_submission


## Q2. Transformation (`custom_transform` in `utils.py`)
WordNet synonym replacement: words longer than 3 letters, probability 0.6, first WordNet sense, capitalized words skipped.

In [10]:
path = "/content/drive/MyDrive/dsga1011/nyu-dsga1011-a4/part1/utils.py"
src = open(path).read()

new_block = '''    ################################
    ##### YOUR CODE BEGINGS HERE ###

    # Synonym replacement: each word longer than 3 letters is replaced, with
    # probability 0.6, by a WordNet synonym from its most common sense.
    def get_synonym(word):
        if word[0].isupper():                          # skip names like "Hanks"
            return word
        candidates = []
        for syn in wordnet.synsets(word)[:1]:
            for lemma in syn.lemmas():
                name = lemma.name().replace("_", " ")
                if name.lower() != word.lower():       # "Movie" -> "movie" isn't a synonym
                    candidates.append(name)
        if len(candidates) == 0:
            return word
        return random.choice(candidates)

    words = word_tokenize(example["text"])
    new_words = []
    for w in words:
        if random.random() < 0.6 and len(w) > 3:
            new_words.append(get_synonym(w))
        else:
            new_words.append(w)
    example["text"] = TreebankWordDetokenizer().detokenize(new_words)

    ##### YOUR CODE ENDS HERE ######'''

s = src.index("def custom_transform(")
fn = src[s:]
b = fn.index("    ################################\n    ##### YOUR CODE BEGINGS HERE ###")
end_tag = "##### YOUR CODE ENDS HERE ######"
f = fn.index(end_tag) + len(end_tag)
src = src[:s] + fn[:b] + new_block + fn[f:]
open(path, "w").write(src)
print("Patched")
!grep -n "random.random() < \|NotImplementedError" {path}

Patched
59:        if random.random() < 0.6 and len(w) > 3:


Examples of transformed reviews

In [12]:
!python3 -u main.py --eval_transformed --debug_transformation

config.json: 100% 570/570 [00:00<00:00, 1.71MB/s]
tokenizer_config.json: 100% 49.0/49.0 [00:00<00:00, 137kB/s]
vocab.txt: 100% 213k/213k [00:00<00:00, 7.72MB/s]
tokenizer.json: 100% 436k/436k [00:00<00:00, 19.4MB/s]
README.md: 100% 7.81k/7.81k [00:00<00:00, 2.55MB/s]

plain_text/train-00000-of-00001.parquet: downloading bytes:  72% 15.0M/21.0M [00:01<00:00, 16.8MB/s,  418kB/s  ]
plain_text/train-00000-of-00001.parquet: downloading bytes: 100% 20.9M/20.9M [00:01<00:00, 16.3MB/s, 2.02MB/s  ]
plain_text/train-00000-of-00001.parquet: reconstructing file: 100% 21.0M/21.0M [00:01<00:00, 16.3MB/s, 2.03MB/s  ]

plain_text/test-00000-of-00001.parquet: downloading bytes:  70% 14.3M/20.5M [00:00<00:00, 28.8MB/s,  285kB/s  ]
plain_text/test-00000-of-00001.parquet: downloading bytes: 100% 20.4M/20.4M [00:00<00:00, 26.2MB/s, 1.99MB/s  ]
plain_text/test-00000-of-00001.parquet: reconstructing file: 100% 20.5M/20.5M [00:00<00:00, 26.3MB/s, 2.01MB/s  ]

plain_text/unsupervised-00000-of-00001.p(…): downl

Evaluation on the transformed test set → `out_transformed.txt`

In [15]:
!python3 -u main.py --eval_transformed

Actual training...
len(train_dataloader): 3125
len(eval_dataloader): 3125
Map: 100% 25000/25000 [02:01<00:00, 205.28 examples/s]
Map: 100% 25000/25000 [00:22<00:00, 1094.37 examples/s]
Loading weights: 100% 201/201 [00:00<00:00, 2571.55it/s]
100% 3125/3125 [12:09<00:00,  4.28it/s]
Score:  {'accuracy': 0.88884}


## Q3. Data augmentation (`create_augmented_dataloader` in `main.py`)

In [65]:
%cd /content/drive/MyDrive/dsga1011/nyu-dsga1011-a4/part1
path = "main.py"
src = open(path).read()
s = src.index("def create_augmented_dataloader(")
e = src.index("def create_transformed_dataloader(")
fn = src[s:e]
fn = fn.replace("    raise NotImplementedError\n", '''    sampled = dataset["train"].shuffle(seed=42).select(range(5000))
    sampled_transformed = sampled.map(custom_transform, load_from_cache_file=False)
    augmented = datasets.concatenate_datasets([dataset["train"], sampled_transformed])

    augmented_tokenized = augmented.map(tokenize_function, batched=True, load_from_cache_file=False)
    augmented_tokenized = augmented_tokenized.remove_columns(["text"])
    augmented_tokenized = augmented_tokenized.rename_column("label", "labels")
    augmented_tokenized.set_format("torch")

    train_dataloader = DataLoader(augmented_tokenized, shuffle=True, batch_size=args.batch_size)
''', 1)
open(path, "w").write(src[:s] + fn + src[e:])

print("=== main.py ===")
!grep -n 'concatenate_datasets\|NotImplementedError' main.py
print("=== utils.py ===")
!grep -n 'def get_synonym\|random.random() <\|NotImplementedError' utils.py

/content/drive/MyDrive/dsga1011/nyu-dsga1011-a4/part1
=== main.py ===
137:    augmented = datasets.concatenate_datasets([dataset["train"], sampled_transformed])
=== utils.py ===
43:    def get_synonym(word):
59:        if random.random() < 0.6 and len(w) > 3:


Train on original + 5,000 transformed examples, evaluate on transformed test set

In [66]:
!python3 main.py --train_augmented --eval_transformed

Actual training...
len(train_dataloader): 3125
len(eval_dataloader): 3125
Map: 100% 5000/5000 [00:40<00:00, 124.87 examples/s]
Map: 100% 30000/30000 [00:29<00:00, 1023.61 examples/s]
Loading weights: 100% 199/199 [00:00<00:00, 4584.06it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ign

Evaluate on original test set → `out_augmented_original.txt`

In [67]:
!python3 main.py --eval --model_dir ./out_augmented

Actual training...
len(train_dataloader): 3125
len(eval_dataloader): 3125
Loading weights: 100% 201/201 [00:00<00:00, 2183.45it/s]
100% 3125/3125 [13:08<00:00,  3.96it/s]
Score:  {'accuracy': 0.92788}


Evaluate on transformed test set → `out_augmented_transformed.txt` (final file)

In [70]:
%cd /content/drive/MyDrive/dsga1011/nyu-dsga1011-a4/part1
!ls -d out_augmented && ls out_augmented_*.txt
!python3 main.py --eval_transformed --model_dir ./out_augmented

/content/drive/MyDrive/dsga1011/nyu-dsga1011-a4/part1
out_augmented
out_augmented_original.txt  out_augmented_transformed.txt
Actual training...
len(train_dataloader): 3125
len(eval_dataloader): 3125
Map: 100% 25000/25000 [02:10<00:00, 190.94 examples/s]
Map: 100% 25000/25000 [00:24<00:00, 1029.40 examples/s]
Loading weights: 100% 201/201 [00:00<00:00, 2682.40it/s]
100% 3125/3125 [13:11<00:00,  3.95it/s]
Score:  {'accuracy': 0.91168}
